In [1]:
!pip install -q transformers datasets accelerate

In [2]:
from transformers import AutoModelForSequenceClassification

# Load BERT and tell it we are doing binary classification (2 labels)
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load the dataset
raw_dataset = load_dataset("stanfordnlp/sst2")

# Load the tokenizer corresponding to the BERT model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Define a tokenization function (remove padding here)
def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True)

# Apply the tokenization to the entire dataset
tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [4]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1821
    })
})


In [5]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

training_args = TrainingArguments(
    output_dir="./bert_sentiment_model",
    learning_rate=2e-5,
    per_device_train_batch_size=32,      # Increased batch size
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=500,                   # Stopped logging every single step
    fp16=True,                           # Enabled mixed-precision training (CRITICAL for speed)
    report_to="none"
)

# Initialize the DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    data_collator=data_collator, # Add the data collator here
    # validation_dataset=tokenized_dataset['validation'],
    # test_dataset=tokenized_dataset['test']
)

# Start the training loop!
trainer.train()

Step,Training Loss
500,0.299504
1000,0.207858
1500,0.184354
2000,0.165609
2500,0.121486
3000,0.105446
3500,0.109003
4000,0.110382
4500,0.083631
5000,0.069634


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6315, training_loss=0.1295436040617036, metrics={'train_runtime': 1205.6956, 'train_samples_per_second': 167.577, 'train_steps_per_second': 5.238, 'total_flos': 4176656240000220.0, 'train_loss': 0.1295436040617036, 'epoch': 3.0})

In [6]:
# Save the trained model
trainer.save_model("./final_bert_sentiment_model")
print("Model saved to ./final_bert_sentiment_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./final_bert_sentiment_model
